In [65]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [98]:
# load titanic dataset
df = sns.load_dataset('titanic')

##  **train_test_split** 하이퍼파라미터
- test_size
- train_size
- random_state
- shuffle: True, False 데이터 분할 전에 무작위로 섞을지 여부
- stratify: 분류문제에서 클래스 비율을 유지한 채로 분할하고 싶을 때,
  - stratify=y 처럼 타겟 벡터를 그대로 넣으면, 학습셋/테스트셋에서 클래스 비율이 유지. 즉, 타겟변수 층화 샘플링도 가능
    

In [99]:
## sex 컬럼 전처리
df['sex'] = df['sex'].map({'male':0, 'female':1})

## embarked 컬럼 One-Hot Encoding
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

In [69]:
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'class',
       'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone',
       'embarked_Q', 'embarked_S'],
      dtype='object')


In [70]:
## train dataset 정리
X = df[['sex', 'age', 'fare', 'embarked_Q', 'embarked_S']]
y = df['survived']

In [71]:
## train_test_split 적용
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2
)

In [72]:
## train data
print(X_train, y_train)

     sex   age     fare  embarked_Q  embarked_S
430  NaN  28.0  26.5500       False        True
455  NaN  29.0   7.8958       False       False
259  NaN  50.0  26.0000       False        True
795  NaN  39.0  13.0000       False        True
573  NaN   NaN   7.7500        True       False
..   ...   ...      ...         ...         ...
346  NaN  40.0  13.0000       False        True
753  NaN  23.0   7.8958       False        True
1    NaN  38.0  71.2833       False       False
400  NaN  39.0   7.9250       False        True
572  NaN  36.0  26.3875       False        True

[712 rows x 5 columns] 430    1
455    1
259    1
795    0
573    1
      ..
346    1
753    0
1      1
400    1
572    1
Name: survived, Length: 712, dtype: int64


- X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
- print(X_train, y_train, y_test)

## 데이터 스케일링
- 모델 학습 전에 입력 변수의 분포를 표준화
- StandardScaler(): 평균이 0, 표준편차가 1인 정규분포로 데이터를 변환


In [73]:
## Scaling 진행
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [74]:
## y 전체의 타겟변수 클래스별 빈도를 비율로 반환
print(y.value_counts(normalize=True))

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [75]:
## y_train의 클래스별 빈도를 비율로 반환
## 분할 후에도 전체 데이터와 유사한 비율로 클래스가 유지되는지 확인
print(y_train.value_counts(normalize=True))

survived
0    0.613764
1    0.386236
Name: proportion, dtype: float64


In [76]:
## y_test 데이터셋의 클래스별 빈도 비율
## 클래스 불균형이 유지되었는지 확인
print(y_test.value_counts(normalize=True))

survived
0    0.625698
1    0.374302
Name: proportion, dtype: float64


- train_test_split 데이터를 자르는 방법, 어떻게 데이터를 자르는가?

## 교차검증 Cross Validation
- 반복적으로 교차 검증을 하는 것
- 위에 train_test_split와 비슷하지만 다른 뜻을 의미한다.

In [77]:
## 교차검증
## k-fold / Sk-fold
from sklearn.model_selection import KFold, StratifiedKFold

In [78]:
## fold를 가지고 올 객체 생성
kf = KFold(n_splits = 5)
skf = StratifiedKFold(n_splits = 5)

In [79]:
X

,sex,age,fare,embarked_Q,embarked_S
0,NaN,22.0,7.2500,False,True
1,NaN,38.0,71.2833,False,False
2,NaN,26.0,7.9250,False,True
3,NaN,35.0,53.1000,False,True
4,NaN,35.0,8.0500,False,True
...,...,...,...,...,...
886,NaN,27.0,13.0000,False,True
887,NaN,19.0,30.0000,False,True
888,NaN,NaN,23.4500,False,True
889,NaN,26.0,30.0000,False,False


## KFold, StratifiedKFold로 나눈 각 test dataset의 클래스 분포 확인
- kf.split(X): X 데이터를 5개의 폴드로 나누고 각 반복마다 train_idx, test_idx 생성
- y.iloc[test_idx]: 테스트 세트의 타겟값(y)들을 가져옴
- Counter(): 테스트세트에서 클래스가 몇 개씩 있는지 개수 세기
- 결과: 테스트 세트마다의 클래스 분포 리스트

In [80]:
from collections import Counter
kf_target_dis = [Counter(y.iloc[test_idx]) for _, test_idx in kf.split(X)]

- skf.split(X, y): y의 클래스 비율을 유지해서 폴드를 나눔
- Counter(): 각 테스트 세트에 클래스가 균형있게 들어가 있는지 확인

In [81]:
skf_target_dis = [Counter(y.iloc[test_idx]) for _, test_idx in skf.split(X,y)]

In [82]:
kf_target_dis

[Counter({0: 120, 1: 59}),
 Counter({0: 99, 1: 79}),
 Counter({0: 109, 1: 69}),
 Counter({1: 72, 0: 106}),
 Counter({0: 115, 1: 63})]

In [83]:
skf_target_dis

[Counter({0: 110, 1: 69}),
 Counter({0: 110, 1: 68}),
 Counter({1: 68, 0: 110}),
 Counter({1: 68, 0: 110}),
 Counter({1: 69, 0: 109})]

## 교차검증
- kf, skf ->데이터셋을 나눠서 진행한다.
- 교차검증을 반복문으로 진행해도 되고, cross_val_score Sklearn에서 제공하는 패키지로도 가능하다!

In [84]:
X

,sex,age,fare,embarked_Q,embarked_S
0,NaN,22.0,7.2500,False,True
1,NaN,38.0,71.2833,False,False
2,NaN,26.0,7.9250,False,True
3,NaN,35.0,53.1000,False,True
4,NaN,35.0,8.0500,False,True
...,...,...,...,...,...
886,NaN,27.0,13.0000,False,True
887,NaN,19.0,30.0000,False,True
888,NaN,NaN,23.4500,False,True
889,NaN,26.0,30.0000,False,False


In [110]:
df = sns.load_dataset('titanic')

In [111]:
df['sex'] = df['sex'].map({'male':0, 'female':1})
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

In [112]:
df = df.dropna(subset=['age'])

In [113]:
X = df[['sex','age','fare','embarked_Q','embarked_S']]
y = df['survived']

In [114]:
df

,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,deck,embark_town,alive,alone,embarked_Q,embarked_S
0,0,3,0,22.0,1,0,7.2500,Third,man,True,NaN,Southampton,no,False,False,True
1,1,1,1,38.0,1,0,71.2833,First,woman,False,C,Cherbourg,yes,False,False,False
2,1,3,1,26.0,0,0,7.9250,Third,woman,False,NaN,Southampton,yes,True,False,True
3,1,1,1,35.0,1,0,53.1000,First,woman,False,C,Southampton,yes,False,False,True
4,0,3,0,35.0,0,0,8.0500,Third,man,True,NaN,Southampton,no,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,1,39.0,0,5,29.1250,Third,woman,False,NaN,Queenstown,no,False,True,False
886,0,2,0,27.0,0,0,13.0000,Second,man,True,NaN,Southampton,no,True,False,True
887,1,1,1,19.0,0,0,30.0000,First,woman,False,B,Southampton,yes,True,False,True
889,1,1,0,26.0,0,0,30.0000,First,man,True,C,Cherbourg,yes,True,False,False


In [115]:
## 패키지를 불러오자!
from sklearn.model_selection import cross_val_score
# cross_val_score(model, X,y, cv=교차검증, scoring='평가지표')

## 모델도 불러오기
from sklearn.linear_model import LogisticRegression

## 로지스틱 베이스 모델 불러오기
model= LogisticRegression(max_iter=200)

## Kfold 교차검증
kf_scores =cross_val_score(model, X, y ,cv=kf, scoring='accuracy')

## Skflold 교차검증
skf_scores =cross_val_score(model, X, y ,cv=skf, scoring='accuracy')

In [116]:
kf_scores

array([0.79020979, 0.7972028 , 0.74825175, 0.73426573, 0.8028169 ])

In [117]:
skf_scores

array([0.72727273, 0.81818182, 0.75524476, 0.73426573, 0.8028169 ])

In [118]:
## 교차검증 cv값을 조정할 수 있다.
base_scores =cross_val_score(model, X, y ,cv=5, scoring='accuracy')

In [119]:
base_scores

array([0.72727273, 0.81818182, 0.75524476, 0.73426573, 0.8028169 ])

### 교차검증을 모두 다 정리해서 확인해 보자!

In [120]:
## 평가지표 불러오기
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [121]:
## 타이타닉 데이터셋 로드
df = sns.load_dataset('titanic')

df=df.dropna(subset=['age'])

df['sex'] = df['sex'].map({'male':0, 'female':1})
df=pd.get_dummies(df, columns=['embarked'], drop_first=True)

## 학습데이터셋 정리

X = df[['sex','age','fare','embarked_Q','embarked_S']]
y = df['survived']


## 처음은 train_test_split 평가

X_train, X_test, y_train, y_test=train_test_split(
    X,y,
    test_size= 0.2,
    random_state=111)


## 스케일링 진행
scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## 로지스틱 베이스 모델 불러오기
model= LogisticRegression(max_iter=200)

## Train, test 결과
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## train,test 성능평가
accuracy_train_test = accuracy_score(y_test, y_pred)
precision_train_test = precision_score(y_test, y_pred)
recall_train_test = recall_score(y_test, y_pred)
f1_train_test = f1_score(y_test, y_pred)

## 전체 타겟 데이터 분포
overall_target_dist = Counter(y)

## train/test로 분할 했을 때 타겟 분포
train_target_dist = Counter(y_train)
test_target_dist = Counter(y_test)

## Kfold 교차 검증
kf = KFold(n_splits = 5, shuffle=True, random_state=111)
kf_accuracy_scores = cross_val_score(model, X, y ,cv=kf, scoring='accuracy')
kf_precision_scores = cross_val_score(model, X, y ,cv=kf, scoring='precision')
kf_recall_scores = cross_val_score(model, X, y ,cv=kf, scoring='recall')
kf_f1_scores = cross_val_score(model, X, y ,cv=kf, scoring='f1')
#kf 타겟 분포
kf_target_dist = [Counter(y.iloc[test_idx]) for _, test_idx in kf.split(X)]


## skf 교차검증
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=111)
skf_accuracy_scores = cross_val_score(model, X, y ,cv=skf, scoring='accuracy')
skf_precision_scores = cross_val_score(model, X, y ,cv=skf, scoring='precision')
skf_recall_scores = cross_val_score(model, X, y ,cv=skf, scoring='recall')
skf_f1_scores = cross_val_score(model, X, y ,cv=skf, scoring='f1')

## Skf 타겟 분포
skf_target_dist = [Counter(y.iloc[test_idx]) for _, test_idx in skf.split(X,y)]

In [122]:
## 결과 최종 정리

res = {
    '전체 타겟 분포':overall_target_dist,
    'train/test':{
        'acc':accuracy_train_test,
        'precision':precision_train_test,
        'recall':recall_train_test,
        'f1':f1_train_test,
        'train 타겟 분포':train_target_dist,
        'test 타겟 분포':test_target_dist
    },

        'KFold':{
        'acc':kf_accuracy_scores,
        'precision':kf_precision_scores,
        'recall':kf_recall_scores,
        'f1':kf_f1_scores,
        'KF 타겟 분포':kf_target_dist,
    },
        'SKFold':{
        'acc':skf_accuracy_scores,
        'precision':skf_precision_scores,
        'recall':skf_recall_scores,
        'f1':skf_f1_scores,
        'SKF 타겟 분포':skf_target_dist,
        }
    }


In [123]:
res

{'전체 타겟 분포': Counter({0: 424, 1: 290}),
 'train/test': {'acc': 0.7482517482517482,
  'precision': 0.6610169491525424,
  'recall': 0.7090909090909091,
  'f1': 0.6842105263157895,
  'train 타겟 분포': Counter({0: 336, 1: 235}),
  'test 타겟 분포': Counter({0: 88, 1: 55})},
 'KFold': {'acc': array([0.74825175, 0.78321678, 0.85314685, 0.74125874, 0.75352113]),
  'precision': array([0.66101695, 0.78571429, 0.88888889, 0.70588235, 0.65384615]),
  'recall': array([0.70909091, 0.6984127 , 0.76190476, 0.62068966, 0.66666667]),
  'f1': array([0.68421053, 0.7394958 , 0.82051282, 0.66055046, 0.66019417]),
  'KF 타겟 분포': [Counter({1: 55, 0: 88}),
   Counter({0: 80, 1: 63}),
   Counter({1: 63, 0: 80}),
   Counter({1: 58, 0: 85}),
   Counter({1: 51, 0: 91})]},
 'SKFold': {'acc': array([0.77622378, 0.79020979, 0.74125874, 0.77622378, 0.78873239]),
  'precision': array([0.75      , 0.73333333, 0.69811321, 0.77083333, 0.74137931]),
  'recall': array([0.67241379, 0.75862069, 0.63793103, 0.63793103, 0.74137931]),
